In [1]:
import pandas as pd
from Bio.PDB import PDBParser, PDBIO, Select
from esm.utils.structure.protein_chain import ProteinChain

df = pd.read_csv('/home/sareeves/PSLMs/data/domainome1/SupplementaryTable5.txt', sep='\t')
df = df.loc[~df['uniprot_ID_mutation'].str.endswith('*')]
df

,domain_ID,uniprot_ID,uniprot_ID_mutation,aa_seq,fitness,fitness_sigma,scaled_fitness,scaled_fitness_sigma,ESM1v_domain,RaSP,...,EVE,Tranception,EVE_domain,rsasa,thermoMPNN,AlphaMissense,ESM1v_full-length,Organism,Gene Names (primary),Gene Names (synonym)
1,A0A2R8Y422_PF00240_2,A0A2R8Y422,A0A2R8Y422_A28C,QIFVKTLMGKTITLEVELSDTIDNVKCKIQDKEGIPPDQQRLIFAG...,0.091262,0.003764,0.025738,0.052712,-9.862297,0.944872,...,NaN,NaN,4.980072,41.3,-0.120494,0.3340,NaN,Homo sapiens (Human),NaN,NaN
2,A0A2R8Y422_PF00240_2,A0A2R8Y422,A0A2R8Y422_A28D,QIFVKTLMGKTITLEVELSDTIDNVKDKIQDKEGIPPDQQRLIFAG...,0.097388,0.003962,0.111544,0.055491,-8.525545,0.344768,...,NaN,NaN,3.766518,41.3,0.842978,0.2790,NaN,Homo sapiens (Human),NaN,NaN
3,A0A2R8Y422_PF00240_2,A0A2R8Y422,A0A2R8Y422_A28E,QIFVKTLMGKTITLEVELSDTIDNVKEKIQDKEGIPPDQQRLIFAG...,0.110153,0.007189,0.290325,0.100693,-7.011676,-0.044519,...,NaN,NaN,2.854782,41.3,0.355797,0.2014,NaN,Homo sapiens (Human),NaN,NaN
4,A0A2R8Y422_PF00240_2,A0A2R8Y422,A0A2R8Y422_A28F,QIFVKTLMGKTITLEVELSDTIDNVKFKIQDKEGIPPDQQRLIFAG...,0.089583,0.003577,0.002225,0.050106,-10.680157,-0.423373,...,NaN,NaN,5.208206,41.3,0.118878,0.3648,NaN,Homo sapiens (Human),NaN,NaN
5,A0A2R8Y422_PF00240_2,A0A2R8Y422,A0A2R8Y422_A28G,QIFVKTLMGKTITLEVELSDTIDNVKGKIQDKEGIPPDQQRLIFAG...,0.087045,0.003294,-0.033329,0.046134,-7.925185,0.662229,...,NaN,NaN,4.270241,41.3,0.776435,0.1135,NaN,Homo sapiens (Human),NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
607076,Q9Y6V0_PF05715_1058,Q9Y6V0,Q9Y6V0_W1104S,TCPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,0.105455,0.012201,-0.029589,0.137867,-5.345161,5.397289,...,NaN,NaN,8.343338,11.8,2.117514,0.9588,-12.64865,Homo sapiens (Human),PCLO,ACZ KIAA0559
607077,Q9Y6V0_PF05715_1058,Q9Y6V0,Q9Y6V0_W1104T,TCPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,0.090854,0.015305,-0.194564,0.172938,-5.214868,4.558003,...,NaN,NaN,9.494637,11.8,1.988062,0.9647,-14.00641,Homo sapiens (Human),PCLO,ACZ KIAA0559
607078,Q9Y6V0_PF05715_1058,Q9Y6V0,Q9Y6V0_W1104V,TCPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,0.115816,0.010449,0.087485,0.118070,-4.870904,3.148088,...,NaN,NaN,8.973549,11.8,1.270481,0.9512,-13.55418,Homo sapiens (Human),PCLO,ACZ KIAA0559
607079,Q9Y6V0_PF05715_1058,Q9Y6V0,Q9Y6V0_W1104Y,TCPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,0.064475,0.022245,-0.492635,0.251357,-4.390252,1.411740,...,NaN,NaN,8.418221,11.8,0.628391,0.8258,-11.89384,Homo sapiens (Human),PCLO,ACZ KIAA0559


In [2]:
len(df['domain_ID'].unique())

522

In [3]:
def align_sequences(seq1, seq2, max_mismatches=1):
    """
    Find the largest aligned region between seq1 and seq2 (seq2 is a subset of seq1),
    allowing for gaps only at the ends of seq1.

    Args:
        seq1 (str): The longer sequence.
        seq2 (str): The shorter sequence (subset of seq1).
        max_mismatches (int): Maximum allowed mismatches in the alignment.

    Returns:
        Tuple[str, str, int, int]: Aligned seq1, aligned seq2, start index, end index.
    """
    best_start = -1
    best_end = -1
    best_mismatches = float('inf')
    best_aligned_seq1 = ""
    best_aligned_seq2 = ""

    for start in range(len(seq1) - len(seq2) + 1):
        # Extract the region of seq1 to compare with seq2
        subseq1 = seq1[start:start + len(seq2)]

        # Count mismatches
        mismatches = sum(1 for a, b in zip(subseq1, seq2) if a != b)

        # Update best alignment if mismatches are within limit and alignment is better
        if mismatches <= max_mismatches and (best_end - best_start) < (start + len(seq2) - start):
            best_start = start
            best_end = start + len(seq2)
            best_mismatches = mismatches
            best_aligned_seq1 = subseq1
            best_aligned_seq2 = seq2

    # Add gaps to the ends of seq1
    aligned_seq1 = ("-" * best_start) + best_aligned_seq1 + ("-" * (len(seq1) - best_end))
    aligned_seq2 = (" " * best_start) + best_aligned_seq2 + (" " * (len(seq1) - best_end))

    return aligned_seq1, aligned_seq2, best_start, best_end

In [4]:
import requests
import os
import glob
from tqdm.notebook import tqdm

def assert_diff_by_one(str1, str2):
    assert len(str1) == len(str2), "Strings are not of same length"
    diff_count = 0
    for i in range(min(len(str1), len(str2))):
        if str1[i] != str2[i]:
            diff_count += 1
    assert diff_count == 1, f"Strings are not different by exactly one character\n{str1}\n{str2}"

def get_alphafold_structure(uniprot_id, output_file):
    base_url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb"
    
    response = requests.get(base_url)
    if response.status_code == 200:
        with open(output_file, 'w') as file:
            file.write(response.text)
        print(f"AlphaFold structure for {uniprot_id} saved to {output_file}")
    else:
        print(f"Structure for UniProt ID {uniprot_id} not found. Status code: {response.status_code}")

class ResidueRangeSelect(Select):
    """Custom selector to retain residues within a specific range."""
    def __init__(self, start_residue, end_residue, chain_id=None):
        self.start_residue = start_residue
        self.end_residue = end_residue
        self.chain_id = chain_id  # Optional: specify chain if needed

    def accept_residue(self, residue):
        # Get residue ID
        res_id = residue.get_id()[1]  # Residue number
        if self.chain_id is None or residue.get_full_id()[2] == self.chain_id:
            return self.start_residue <= res_id <= self.end_residue
        return False

def truncate_pdb(input_pdb, output_pdb, start_residue, end_residue, chain_id=None):
    """Truncate a PDB file to retain only residues in the specified range."""
    parser = PDBParser(QUIET=True)  # Suppress warnings
    structure = parser.get_structure("protein", input_pdb)

    # Use the custom selector to filter residues
    io = PDBIO()
    io.set_structure(structure)
    selector = ResidueRangeSelect(start_residue, end_residue, chain_id)
    io.save(output_pdb, selector)
    print(f"Truncated PDB saved to {output_pdb}")

failed_assertions = []
for (domain_id, uniprot_id), group in tqdm(df.groupby(['domain_ID', 'uniprot_ID'])):
    print(domain_id)
    subseq = group['aa_seq'].head(1).item()
    print(subseq)

    file = f"/home/sareeves/PSLMs/data/domainome1/alphafold_structures/{uniprot_id}.pdb"
    if not os.path.exists(file):
        get_alphafold_structure(uniprot_id, file)

    try:
        wt_prot = ProteinChain.from_pdb(file)
    except ValueError as e:
        print(e)
        print(f'Issue with loading {file}')
        continue
    except FileNotFoundError as e:
        print(e)
        print(f'No file for {file}') 
        continue     

    full_seq = wt_prot.sequence
    aligned_seq1, aligned_seq2, start, end = align_sequences(full_seq, subseq, max_mismatches=1)
    start += 1
    #print("Aligned seq1:", aligned_seq1)
    #print("Aligned seq2:", aligned_seq2)
    #print(f"Aligned region starts at {start}, ends at {end}")

    truncated_file = file.replace('.pdb', f'_{start}-{end}.pdb')
    truncate_pdb(file, truncated_file, start, end)
    trunc_prot = ProteinChain.from_pdb(truncated_file)

    #print(trunc_prot.sequence)
    assert_diff_by_one(subseq, trunc_prot.sequence)
    #print(subseq)
    #print(trunc_prot.sequence)

    df.loc[(df['domain_ID']==domain_id), 'pdb_file'] = truncated_file
    for idx, row in group.iterrows():
        wt = row['uniprot_ID_mutation'].split('_')[-1][0]
        try:
            pos = int(row['uniprot_ID_mutation'].split('_')[-1][1:-1])
        except:
            continue
        pos -= start
        mut = row['uniprot_ID_mutation'].split('_')[-1][-1]

        try:
            assert trunc_prot.sequence[pos] == wt
            assert row['aa_seq'][pos] == mut
        except:
            failed_assertions.append((domain_id, wt, pos, mut))
            print(domain_id, wt, pos, mut)
            print(trunc_prot.sequence)
            print(row['aa_seq'])
            continue
        df.at[idx, 'wt_seq'] = trunc_prot.sequence
        df.at[idx, 'wild_type'] = wt
        df.at[idx, 'position'] = pos+1
        df.at[idx, 'mutation'] = mut
        df.at[idx, 'uniprot_seq'] = full_seq
        df.at[idx, 'offset_up'] = -start+1

df.to_csv('../data/domainome1/domainome_mapped.csv')

  0%|          | 0/522 [00:00<?, ?it/s]

A0A2R8Y422_PF00240_2
QIFVKTLMGKTITLEVELSDTIDNVKCKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVL
Truncated PDB saved to /home/sareeves/PSLMs/data/domainome1/alphafold_structures/A0A2R8Y422_2-71.pdb
A0PJY2_PF00096_289
VCKVCGKGFRQCSTLCRHKIIH
Truncated PDB saved to /home/sareeves/PSLMs/data/domainome1/alphafold_structures/A0PJY2_289-310.pdb
A1X283_PF00018_155
EQYVVVANYQKQESSEISLSVGQVVDIIEKNESGWWFVSTCEEQGWVPATCLEGQ
Truncated PDB saved to /home/sareeves/PSLMs/data/domainome1/alphafold_structures/A1X283_155-209.pdb
A1X283_PF00018_222
EEEKYTVIYPYTCRDQDEMNLERGAVVEVIQKNLEGWWKIRYQGKEGWAPASYLKKNSG
Truncated PDB saved to /home/sareeves/PSLMs/data/domainome1/alphafold_structures/A1X283_222-280.pdb
A2RRE5_PF01846_267
SQQIATAKDKYEWLVSRIVKNHNENWLSVSRKMQCSPEYQDYVYLEGTQKAKKLFLQHIHRLK
Truncated PDB saved to /home/sareeves/PSLMs/data/domainome1/alphafold_structures/A2RRE5_267-329.pdb
A6NK59_PF07525_532
RSLKHLCRLKIRKCMGRLHLRCPVFMSFLPLPNRLKCYVLY
Truncated PDB saved to /home/sareeves/PSLMs/data/domainome1/alphafo

In [5]:
failed_assertions

[('Q13472_PF06839_896', 'A', 25, 'C'),
 ('Q13472_PF06839_896', 'A', 25, 'D'),
 ('Q13472_PF06839_896', 'A', 25, 'E'),
 ('Q13472_PF06839_896', 'A', 25, 'F'),
 ('Q13472_PF06839_896', 'A', 25, 'G'),
 ('Q13472_PF06839_896', 'A', 25, 'H'),
 ('Q13472_PF06839_896', 'A', 25, 'I'),
 ('Q13472_PF06839_896', 'A', 25, 'K'),
 ('Q13472_PF06839_896', 'A', 25, 'L'),
 ('Q13472_PF06839_896', 'A', 25, 'M'),
 ('Q13472_PF06839_896', 'A', 25, 'N'),
 ('Q13472_PF06839_896', 'A', 25, 'P'),
 ('Q13472_PF06839_896', 'A', 25, 'Q'),
 ('Q13472_PF06839_896', 'A', 25, 'R'),
 ('Q13472_PF06839_896', 'A', 25, 'S'),
 ('Q13472_PF06839_896', 'A', 25, 'T'),
 ('Q13472_PF06839_896', 'A', 25, 'V'),
 ('Q13472_PF06839_896', 'A', 25, 'W'),
 ('Q13472_PF06839_896', 'A', 25, 'Y'),
 ('Q13472_PF06839_896', 'C', -1, 'A'),
 ('Q13472_PF06839_896', 'C', -1, 'D'),
 ('Q13472_PF06839_896', 'C', -1, 'E'),
 ('Q13472_PF06839_896', 'C', -1, 'F'),
 ('Q13472_PF06839_896', 'C', -1, 'G'),
 ('Q13472_PF06839_896', 'C', -1, 'H'),
 ('Q13472_PF06839_896', '

In [12]:
import pandas as pd
df = pd.read_csv('../data/domainome1/domainome_mapped.csv', index_col=0)
df

,domain_ID,uniprot_ID,uniprot_ID_mutation,aa_seq,fitness,fitness_sigma,scaled_fitness,scaled_fitness_sigma,ESM1v_domain,RaSP,...,Organism,Gene Names (primary),Gene Names (synonym),pdb_file,wt_seq,wild_type,position,mutation,uniprot_seq,offset_up
1,A0A2R8Y422_PF00240_2,A0A2R8Y422,A0A2R8Y422_A28C,QIFVKTLMGKTITLEVELSDTIDNVKCKIQDKEGIPPDQQRLIFAG...,0.091262,0.003764,0.025738,0.052712,-9.862297,0.944872,...,Homo sapiens (Human),NaN,NaN,/home/sareeves/PSLMs/data/domainome1/alphafold...,QIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,A,27.0,C,MQIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFA...,-1.0
2,A0A2R8Y422_PF00240_2,A0A2R8Y422,A0A2R8Y422_A28D,QIFVKTLMGKTITLEVELSDTIDNVKDKIQDKEGIPPDQQRLIFAG...,0.097388,0.003962,0.111544,0.055491,-8.525545,0.344768,...,Homo sapiens (Human),NaN,NaN,/home/sareeves/PSLMs/data/domainome1/alphafold...,QIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,A,27.0,D,MQIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFA...,-1.0
3,A0A2R8Y422_PF00240_2,A0A2R8Y422,A0A2R8Y422_A28E,QIFVKTLMGKTITLEVELSDTIDNVKEKIQDKEGIPPDQQRLIFAG...,0.110153,0.007189,0.290325,0.100693,-7.011676,-0.044519,...,Homo sapiens (Human),NaN,NaN,/home/sareeves/PSLMs/data/domainome1/alphafold...,QIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,A,27.0,E,MQIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFA...,-1.0
4,A0A2R8Y422_PF00240_2,A0A2R8Y422,A0A2R8Y422_A28F,QIFVKTLMGKTITLEVELSDTIDNVKFKIQDKEGIPPDQQRLIFAG...,0.089583,0.003577,0.002225,0.050106,-10.680157,-0.423373,...,Homo sapiens (Human),NaN,NaN,/home/sareeves/PSLMs/data/domainome1/alphafold...,QIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,A,27.0,F,MQIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFA...,-1.0
5,A0A2R8Y422_PF00240_2,A0A2R8Y422,A0A2R8Y422_A28G,QIFVKTLMGKTITLEVELSDTIDNVKGKIQDKEGIPPDQQRLIFAG...,0.087045,0.003294,-0.033329,0.046134,-7.925185,0.662229,...,Homo sapiens (Human),NaN,NaN,/home/sareeves/PSLMs/data/domainome1/alphafold...,QIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,A,27.0,G,MQIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFA...,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
607076,Q9Y6V0_PF05715_1058,Q9Y6V0,Q9Y6V0_W1104S,TCPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,0.105455,0.012201,-0.029589,0.137867,-5.345161,5.397289,...,Homo sapiens (Human),PCLO,ACZ KIAA0559,NaN,NaN,NaN,NaN,NaN,NaN,NaN
607077,Q9Y6V0_PF05715_1058,Q9Y6V0,Q9Y6V0_W1104T,TCPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,0.090854,0.015305,-0.194564,0.172938,-5.214868,4.558003,...,Homo sapiens (Human),PCLO,ACZ KIAA0559,NaN,NaN,NaN,NaN,NaN,NaN,NaN
607078,Q9Y6V0_PF05715_1058,Q9Y6V0,Q9Y6V0_W1104V,TCPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,0.115816,0.010449,0.087485,0.118070,-4.870904,3.148088,...,Homo sapiens (Human),PCLO,ACZ KIAA0559,NaN,NaN,NaN,NaN,NaN,NaN,NaN
607079,Q9Y6V0_PF05715_1058,Q9Y6V0,Q9Y6V0_W1104Y,TCPLCKTELNIGSKDPPNFNTCTECKNQVCNLCGFNPTPHLTEIQE...,0.064475,0.022245,-0.492635,0.251357,-4.390252,1.411740,...,Homo sapiens (Human),PCLO,ACZ KIAA0559,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df_filtered = df.dropna(subset=['position', 'pdb_file', 'scaled_fitness'])
df_filtered = df_filtered.loc[~df_filtered['uniprot_ID_mutation'].str.contains('NANANA')]
df_filtered['position'] = df_filtered['position'].astype(int)
df_filtered['code'] = df_filtered['domain_ID']
df_filtered['chain'] = 'A'
df_filtered['mut_type'] = df_filtered['wild_type'] + df_filtered['position'].astype(str) + df_filtered['mutation']
df_filtered['ddG_ML'] = df_filtered['scaled_fitness']
df_filtered = df_filtered[['uniprot_ID_mutation', 'code', 'chain', 'mut_type', 'wild_type', 'position', 'mutation', 'aa_seq', 'pdb_file', 'ddG_ML']]
df_filtered.to_csv('../data/domainome1/domainome_mapped_filtered.csv')
df_filtered

,uniprot_ID_mutation,code,chain,mut_type,wild_type,position,mutation,aa_seq,pdb_file,ddG_ML
1,A0A2R8Y422_A28C,A0A2R8Y422_PF00240_2,A,A27C,A,27,C,QIFVKTLMGKTITLEVELSDTIDNVKCKIQDKEGIPPDQQRLIFAG...,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.025738
2,A0A2R8Y422_A28D,A0A2R8Y422_PF00240_2,A,A27D,A,27,D,QIFVKTLMGKTITLEVELSDTIDNVKDKIQDKEGIPPDQQRLIFAG...,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.111544
3,A0A2R8Y422_A28E,A0A2R8Y422_PF00240_2,A,A27E,A,27,E,QIFVKTLMGKTITLEVELSDTIDNVKEKIQDKEGIPPDQQRLIFAG...,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.290325
4,A0A2R8Y422_A28F,A0A2R8Y422_PF00240_2,A,A27F,A,27,F,QIFVKTLMGKTITLEVELSDTIDNVKFKIQDKEGIPPDQQRLIFAG...,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.002225
5,A0A2R8Y422_A28G,A0A2R8Y422_PF00240_2,A,A27G,A,27,G,QIFVKTLMGKTITLEVELSDTIDNVKGKIQDKEGIPPDQQRLIFAG...,/home/sareeves/PSLMs/data/domainome1/alphafold...,-0.033329
...,...,...,...,...,...,...,...,...,...,...
605839,Q9Y6N9_V289R,Q9Y6N9_PF00595_207,A,V83R,V,83,R,NKEKKVFISLVGSRGLGCSISSGPIQKPGIFISHVKPGSLSAEVGL...,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.040825
605840,Q9Y6N9_V289S,Q9Y6N9_PF00595_207,A,V83S,V,83,S,NKEKKVFISLVGSRGLGCSISSGPIQKPGIFISHVKPGSLSAEVGL...,/home/sareeves/PSLMs/data/domainome1/alphafold...,-0.228931
605841,Q9Y6N9_V289T,Q9Y6N9_PF00595_207,A,V83T,V,83,T,NKEKKVFISLVGSRGLGCSISSGPIQKPGIFISHVKPGSLSAEVGL...,/home/sareeves/PSLMs/data/domainome1/alphafold...,-0.353383
605842,Q9Y6N9_V289W,Q9Y6N9_PF00595_207,A,V83W,V,83,W,NKEKKVFISLVGSRGLGCSISSGPIQKPGIFISHVKPGSLSAEVGL...,/home/sareeves/PSLMs/data/domainome1/alphafold...,-0.185128


In [4]:
with open('../data/domainome1/domainome_seq.fasta', 'w') as f:
    for (domain, wt_seq), group in df.groupby(['domain_ID', 'wt_seq']):
        print(domain, wt_seq)
        f.write(f'>{domain}\n')
        f.write(f'{wt_seq}\n')

A0A2R8Y422_PF00240_2 QIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVL
A0PJY2_PF00096_289 VCKVCGKGFRQASTLCRHKIIH
A1X283_PF00018_155 EQYVVVANYQKQESSEISLSVGQVVDIIEKNESGWWFVSTAEEQGWVPATCLEGQ
A1X283_PF00018_222 EEEKYTVIYPYTARDQDEMNLERGAVVEVIQKNLEGWWKIRYQGKEGWAPASYLKKNSG
A2RRE5_PF01846_267 SQQIATAKDKYEWLVSRIVKNHNENWLSVSRKMQASPEYQDYVYLEGTQKAKKLFLQHIHRLK
A6NK59_PF07525_532 RSLKHLCRLKIRKCMGRLHLRCPVFMSFLPLPNRLKAYVLY
O00308_PF00397_440 MIQEPALPPGWEMKYTSEGVRYFVDHNTRTTTFKDPRPGFESGTKQGSPGAYDR
O00330_PF02817_181 RFRLSPAARNILEKHSLDASQGTATGPRGIFTKEDALKLVQLKQTGK
O14529_PF02376_1036 GIQEIVAMSPELDTYSITKRVKEVLTDNNLGQRLFGESILGLTQGSVSDLLSRPKPWHKLSLKGREPFVRMQLWLNDPHNVEKLRDMKK
O14529_PF02376_895 EVDTLELTRQVKEKLAKNGICQRIFGEKVLGLSQGSVSDMLSRPKPWSKLTQKGREPFIRMQLWLSDQL
O14640_PF00595_246 SLNIVTVTLNMERHHFLGISIVGQSNDRGDGGIYIGSIMKGGAVAADGRIEPGDMLLQVNDVNFENMSNDDAVRVLREIVSQTGPISLTVAKC
O14640_PF00778_4 TKIIYHMDEEETPYLVKLPVAPERVTLADFKNVLSNRPVHAYKFFFKSMDQDFGVVKEEIFDDNAKLPCFNGRVVSWL
O14776_PF01846_645 

In [ ]:
import pickle

with open('/home/sareeves/software/esm-msr/cache_domainome/cache/A1X283_PF00018_222_ddG_ML_doubles1_mutctx0_rev0_mutS0_Smasked0.pkl', 'rb') as f:
    cf = pickle.load(f)

cf